# **Arquitectura y Entrenamiento del modelo IDS Baseline basado en Machine Learning: LightGBM**

Autor: Daniel Gomollón Embid

Trabajo Fin de Grado:

Fecha: 10/03/2026

**Universidad de Zaragoza**

Este notebook entrena el modelo base (State-of-the-Art en datos tabulares) contra el que competirá nuestro modelo híbrido (Tabular ResNet + VAE).

## **Metodología:**

- Carga de los datos preprocesados (y submuestreados) de BigFlow-NIDS.

- Optimización bayesiana de hiperparámetros mediante Optuna (optimizando el F1-Macro).

- Entrenamiento final del modelo LightGBM.

- Evaluación del modelo aplicando la corrección de probabilidad a priori (Prior Shift Correction) para simular un entorno de producción (95% benigno / 5% ataque).

- Análisis de Importancia de Variables (Gain) y preparación de valores SHAP para mi ataque Gray-box Adversarial Attack.

*Nota de rendimiento: LightGBM construye histogramas por característica, lo que lo hace extremadamente eficiente en CPU (aprox. 5x más rápido que XGBoost tradicional en estas cargas de trabajo).*

## **Importaciones y Configuración Inicial**

In [ ]:
import os
import sys
from pathlib import Path

# --- CONFIGURACIÓN DE PROTECCIÓN DE HARDWARE ---
# Forzamos a usar máximo 6 hilos para proteger el Ryzen 7 5800H
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

project_root = Path(r"C:\Users\Daniel\Desktop\INGENIERÍA INFORMÁTICA\QUINTO AÑO\TFG\Codigo\Codigo_Experimental")
sys.path.insert(0, str(project_root))
os.chdir(project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.models.baseline.baseline_model import LGBMBaseline
from src.models.baseline.lgbm_tuner import LGBMTuner

from src.config import Config
from src.evaluator import ModelEvaluator

from src.models.baseline.baseline_model import LGBMBaseline
from src.config import Config
from src.evaluator import ModelEvaluator

# 3. Rutas y directorios
DATA_PATH  = project_root / "data" / "processed" / "resultados_2_buffer"
OUTPUT_DIR = project_root / "outputs" / "models" / "lgbm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[*] Raíz:   {project_root}")
print(f"[*] Datos:  {DATA_PATH}")
print(f"[*] Existe: {DATA_PATH.exists()}")

-----------------------

## **1. Carga de Datos y Metadatos**
Cargamos los arrays .npy generados por el Data Pipeline. También recuperamos el pi_train (la proporción real de ataques que vio el modelo durante el entrenamiento), vital para corregir las probabilidades en inferencia.

In [ ]:
def load_data(data_path: Path):
    print(f"[-] Cargando datos desde {data_path}...")
    splits = {}
    for split in ('X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test'):
        fpath = data_path / f"{split}.npy"
        if not fpath.exists():
            raise FileNotFoundError(f"Archivo no encontrado: {fpath}. ¿Has ejecutado el pipeline de datos?")
        splits[split] = np.load(fpath)

    return splits['X_train'], splits['y_train'], splits['X_val'], splits['y_val'], splits['X_test'], splits['y_test']

def load_metadata(models_path: Path):
    pi_path = models_path / "pi_train.npy"
    fn_path = models_path / "feature_names.npy"

    pi_train = float(np.load(pi_path)[0]) if pi_path.exists() else 0.40
    feature_names = np.load(fn_path, allow_pickle=True).tolist() if fn_path.exists() else None

    return pi_train, feature_names

# Ejecución
DATA_PATH = project_root / "data" / "processed" / "resultados_2_buffer"
MODELS_PATH = project_root / "outputs" / "models"

X_train, y_train, X_val, y_val, X_test, y_test = load_data(DATA_PATH)
pi_train, feature_names = load_metadata(MODELS_PATH)

# LightGBM usa raw — invertir el escalado
scaler_global = joblib.load(MODELS_PATH / "quantile_scaler_global.pkl")
scaler_benign = joblib.load(MODELS_PATH / "quantile_scaler_benign.pkl")
buf_indices   = np.load(MODELS_PATH / "buf_indices.npy")
other_indices = np.load(MODELS_PATH / "other_indices.npy")

def inverse_scale(X_sc):
    X_raw = np.empty(X_sc.shape, dtype=np.float64)
    X_raw[:, other_indices] = scaler_global.inverse_transform(X_sc[:, other_indices])
    X_raw[:, buf_indices]   = scaler_benign.inverse_transform(X_sc[:, buf_indices])
    return X_raw

X_train = inverse_scale(X_train)
X_val   = inverse_scale(X_val)
X_test  = inverse_scale(X_test)

print(f"\n[✔] Datos cargados e invertidos a escala raw para LightGBM:")
print(f"    Train : {X_train.shape[0]:,} muestras | {X_train.shape[1]} features")
print(f"    Val   : {X_val.shape[0]:,} muestras")
print(f"    Test  : {X_test.shape[0]:,} muestras")
print(f"    Prior de Entrenamiento (π_train): {pi_train:.4f}")

## **2. Optimización de Hiperparámetros (Optuna)**
Buscamos la combinación óptima de hiperparámetros para LightGBM. Nos centramos en parámetros que controlan el sobreajuste (num_leaves, min_child_samples, reg_alpha) y la velocidad de aprendizaje.

- Ajustar N_TRIALS según el tiempo que quera,ps dedicar a esta fase.

In [ ]:
RUN_OPTUNA = True  # Cambiar a False si uso el modelo base rápido
N_TRIALS = 40
TIMEOUT = 3600 # 1h máximo

if RUN_OPTUNA:
    print(f"[*] Iniciando Optimización Optuna ({N_TRIALS} trials, timeout={TIMEOUT}s)...")
    tuner = LGBMTuner(n_trials=N_TRIALS, timeout=TIMEOUT)
    result = tuner.run(X_train, y_train, X_val, y_val)

    print("\n" + result.summary())
    model = result.best_model
else:
    print("[*] Modo Rápido: Usando hiperparámetros por defecto de LightGBM...")
    model = LGBMBaseline()
    model.fit(X_train, y_train, X_val, y_val)

## **3. Evaluación del Modelo y Corrección Prior**
Usamos el ModelEvaluator para medir el rendimiento. Es crucial aplicar la corrección a priori (pi_prod=0.05), asumiendo que en un entorno real de SOC, solo del 1% al 5% del tráfico será malicioso. Esto nos da una métrica de Falsos Positivos realista.

In [ ]:
evaluator = ModelEvaluator(n_classes=8, pi_prod=0.05)

print("[*] Evaluando conjunto de Validación...")
probs_val = model.predict_proba(X_val)
evaluator.evaluate(probs_val, y_val, pi_train, label="Validación")

print("\n[*] Evaluando conjunto de Test...")
probs_test = model.predict_proba(X_test)
result_test = evaluator.evaluate(probs_test, y_test, pi_train, label="Test")

# Guardar predicciones para posibles ensembles o análisis posteriores
np.save(OUTPUT_DIR / "lgbm_test_preds.npy", result_test.preds)
np.save(OUTPUT_DIR / "lgbm_test_probs.npy", result_test.probs)
np.save(OUTPUT_DIR / "lgbm_test_labels.npy", y_test)

print(f"\n[✔] Resultados guardados en {OUTPUT_DIR}")

## **MATRIZ DE CONFUSIÓN LIGHTGBM**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

# ==============================================================================
# 1. CONFIGURACIÓN Y EXTRACTO DE PREDICCIONES
# ==============================================================================
# Taxonomía exacta de tus 8 macroclases funcionales
class_names = ['Benign', 'DoS', 'DDoS', 'Web/Injection', 'Brute Force', 'Recon', 'Malware', 'Exploits']

# Recuperamos las variables. Si ejecutas la celda de forma aislada, las lee del disco
try:
    y_pred = result_test.preds  # Predicciones ya corregidas por el Evaluator (priors)
    y_true = y_test
except NameError:
    y_pred = np.load(OUTPUT_DIR / "lgbm_test_preds.npy")
    y_true = np.load(OUTPUT_DIR / "lgbm_test_labels.npy")

# ==============================================================================
# 2. CÁLCULO DE MATRICES (ABSOLUTA Y NORMALIZADA POR FILA / RECALL)
# ==============================================================================
cm = confusion_matrix(y_true, y_pred)
cm_norm = confusion_matrix(y_true, y_pred, normalize='true') # Proporciones por fila

# ==============================================================================
# 3. GENERACIÓN DEL GRÁFICO ESTILO PAPER ACADÉMICO
# ==============================================================================
plt.figure(figsize=(11, 9), dpi=300)

# Generamos las anotaciones de las celdas: "Valor Absoluto \n (Porcentaje %)"
box_labels = np.asarray([
    f"{val:,}\n({pct:.1%})" if val > 0 else "0"
    for val, pct in zip(cm.flatten(), cm_norm.flatten())
]).reshape(8, 8)

# Mapeado visual con Seaborn
sns.heatmap(
    cm_norm, 
    annot=box_labels, 
    fmt="", 
    cmap="Blues", 
    xticklabels=class_names, 
    yticklabels=class_names,
    square=True,
    cbar_kws={'label': 'Recall'},
    annot_kws={"size": 9}
)

plt.title("Matriz de Confusión - LightGBM", 
          fontsize=12, pad=20, weight='bold')
plt.xlabel("Clase predicha", fontsize=11, labelpad=10)
plt.ylabel("Clase real", fontsize=11, labelpad=10)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Guardado automático para exportar a LaTeX
plt.savefig(OUTPUT_DIR / "lgbm_confusion_matrix.pdf", bbox_inches='tight')
plt.savefig(OUTPUT_DIR / "lgbm_confusion_matrix.png", bbox_inches='tight', dpi=300)
plt.show()

print("[✔] Gráficos de la matriz guardados en formatos PDF y PNG.")

# ==============================================================================
# 4. PROPUESTA EXTRA PARA LA MEMORIA: ANÁLISIS DE FATIGA DE ALERTAS
# ==============================================================================
print("\n" + "="*70)
print("MÉTRICA COMPLEMENTARIA PARA LA MEMORIA: DESTINO DEL TRÁFICO FALSO POSITIVO")
print("="*70)
print("Dirección de las falsas alarmas (Tráfico Benigno clasificado como Ataque):")

benign_mask = (y_true == 0)
fp_benign_pred = y_pred[benign_mask & (y_pred != 0)]
total_benign = benign_mask.sum()

if len(fp_benign_pred) > 0:
    fp_counts = pd.Series(fp_benign_pred).value_counts().sort_index()
    for idx, count in fp_counts.items():
        percentage_of_total_benign = (count / total_benign) * 100
        print(f"  → Alerta Falsa de tipo [{class_names[idx]:<13}]: {count:>6,} flujos "
              f"({percentage_of_total_benign:.4f}% del tráfico benigno total)")
else:
    print("  ¡Felicidades! 0 Falsos Positivos en el conjunto de test bajo este umbral.")

## **4. Interpretabilidad Global (Feature Importance Nativa)**

Analizamos qué características está utilizando el árbol para tomar decisiones. Esto valida nuestro Feature Engineering (como el Buffer o el Proxy HTTP).

In [ ]:
if model.feature_importances_ is not None and feature_names:
    imp = model.feature_importances_
    # Seleccionar top 20
    top_n = 20
    top_idx = np.argsort(imp)[-top_n:]

    names = [feature_names[i] for i in top_idx]
    values = imp[top_idx]

    plt.figure(figsize=(10, 8))
    bars = plt.barh(names, values, color='skyblue')
    plt.xlabel("Importancia Nativa (Gain)", fontsize=12)
    plt.title(f"Top {top_n} Características más importantes (LightGBM)", fontsize=14, fontweight='bold')
    plt.grid(axis='x', linestyle='--', alpha=0.7)

    # Añadir el valor numérico al final de cada barra
    for bar in bars:
        width = bar.get_width()
        plt.text(width + (max(values)*0.01), bar.get_y() + bar.get_height()/2,
                 f'{int(width):,}', ha='left', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "lgbm_native_importance.png", dpi=300)
    plt.show()
else:
    print("[!] Nombres de características no disponibles para plotear.")

## Calibración de umbrales

In [ ]:
# Calibración de umbrales (opcional pero potente)
result_test_cal, threshold_result = evaluator.evaluate_with_thresholds(
    probs_val  = model.predict_proba(X_val),
    y_val      = y_val,
    probs_test = model.predict_proba(X_test),
    y_test     = y_test,
    pi_train   = pi_train,
    label      = "Test calibrado"
)
joblib.dump(threshold_result.thresholds, OUTPUT_DIR / "lgbm_thresholds.pkl")

## **5. Análisis SHAP y Exportación (Preparación para Fase 2)**

Calculamos los valores SHAP (SHapley Additive exPlanations) sobre una muestra del conjunto de prueba.

Estos valores no solo nos dan la interpretabilidad final para la memoria del TFG, sino que serán el núcleo de nuestro generador de Ataques Adversarios de Caja Gris (Feature-Space Perturbation).

Si sabemos qué variables alteran más la decisión del modelo, sabemos dónde atacar.

In [ ]:
import joblib

# Definir la ruta exacta del modelo (usando tu variable OUTPUT_DIR)
model_path = OUTPUT_DIR / "lgbm_baseline.pkl"

print(f"[*] Cargando modelo desde: {model_path} ...")

# Cargar el modelo en la variable 'model'
try:
    model = joblib.load(model_path)
    print("[✔] Modelo cargado en memoria exitosamente.")
except Exception as e:
    print(f"[!] Error al cargar el modelo. Verifica la ruta o el método de guardado. Detalle: {e}")

In [ ]:
RUN_SHAP      = True
N_SAMPLES_SHAP = 20000

if RUN_SHAP:
    try:
        import shap
        print(f"[-] Inicializando SHAP TreeExplainer — "
              f"Muestreo estratégico ({N_SAMPLES_SHAP} muestras, 8 hilos)...")

        # Muestreo estratificado: 10k ataques + 10k benignos
        np.random.seed(Config.SEED)
        idx_attack = np.where(y_test > 0)[0]
        idx_benign = np.where(y_test == 0)[0]

        n_attack = min(10000, len(idx_attack))
        n_benign = min(N_SAMPLES_SHAP - n_attack, len(idx_benign))

        idx_shap = np.concatenate([
            np.random.choice(idx_attack, n_attack, replace=False),
            np.random.choice(idx_benign, n_benign, replace=False),
        ])
        np.random.shuffle(idx_shap)

        X_shap = X_test[idx_shap]
        y_shap = y_test[idx_shap]

        print(f"    Shape X_shap : {X_shap.shape}")
        print(f"    Distribución : Ataques={( y_shap > 0).sum():,} | "
              f"Benignos={(y_shap == 0).sum():,}")
        print(f"    Clases       : {dict(zip(*np.unique(y_shap, return_counts=True)))}")

        # SHAP TreeExplainer
        explainer   = shap.TreeExplainer(model.model)
        shap_values = explainer.shap_values(X_shap, check_additivity=False)

        # Importancia media absoluta agregada sobre todas las clases
        if isinstance(shap_values, list):
            mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
        else:
            mean_abs = np.abs(shap_values).mean(axis=0)

        # ── GUARDAR ARTEFACTOS ────────────────────────────────────────────
        joblib.dump(shap_values, OUTPUT_DIR / "lgbm_shap_values.pkl")      # lista arrays
        joblib.dump(explainer,   OUTPUT_DIR / "lgbm_shap_explainer.pkl")   # objeto sklearn
        np.save(OUTPUT_DIR / "lgbm_shap_mean_abs.npy", mean_abs)           # array simple
        np.save(OUTPUT_DIR / "lgbm_shap_X.npy",        X_shap)            # array simple
        np.save(OUTPUT_DIR / "lgbm_shap_y.npy",        y_shap)            # array simple

        print("\n[✔] SHAP calculado y artefactos guardados.")
        print(f"    lgbm_shap_values.pkl   → {(OUTPUT_DIR / 'lgbm_shap_values.pkl').stat().st_size / 1e6:.1f} MB")
        print(f"    lgbm_shap_explainer.pkl")
        print(f"    lgbm_shap_mean_abs.npy")
        print(f"    lgbm_shap_X.npy  /  lgbm_shap_y.npy")

        # ── SHAP SUMMARY PLOT (Top 20) ────────────────────────────────────
        print("\n[*] Generando SHAP Summary Plot (Top 20)...")
        df_shap = pd.DataFrame(X_shap, columns=feature_names) \
                  if feature_names is not None else pd.DataFrame(X_shap)

        plt.figure(figsize=(10, 8))
        if isinstance(shap_values, list):
            shap.summary_plot(
                shap_values,
                df_shap,
                plot_type  = "bar",
                max_display = 20,
                show        = False,
            )
        else:
            shap.summary_plot(shap_values, df_shap, max_display=20, show=False)

        plt.title("SHAP Feature Importance — Impacto por clase (Top 20)",
                  pad=20, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "lgbm_shap_summary.png", dpi=300, bbox_inches='tight')
        plt.show()
        print(f"[✔] Plot guardado en {OUTPUT_DIR / 'lgbm_shap_summary.png'}")

    except ImportError:
        print("[!] 'shap' no instalado. Ejecuta: pip install shap")

else:
    print("[*] Análisis SHAP omitido (RUN_SHAP=False).")

In [ ]:
print("\n[*] Generando SHAP Summary Plot — Barras por clase (Top 20)...")
df_shap = pd.DataFrame(X_shap, columns=feature_names) \
          if feature_names is not None else pd.DataFrame(X_shap)

from src.models.baseline.baseline_model import CLASS_NAMES
class_names_list = [CLASS_NAMES[i] for i in range(8)]

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    df_shap,
    plot_type    = "bar",
    class_names  = class_names_list,
    max_display  = 20,
    show         = False,
)
plt.title("SHAP Feature Importance — Impacto medio por clase (Top 20)",
          pad=20, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lgbm_shap_summary_bar.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"[✔] Plot guardado.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Paleta Wong (2011) — daltonismo-friendly
WONG_PALETTE = [
    '#0072B2',  # azul         → DDoS
    '#009E73',  # verde teal   → Benign
    '#E69F00',  # naranja      → DoS
    '#CC79A7',  # rosa/morado  → Malware
    '#56B4E9',  # azul claro   → Web/Injection
    '#D55E00',  # naranja rojo → Recon
    '#F0E442',  # amarillo     → Brute Force
    '#000000',  # negro        → Exploits
]

from src.models.baseline.baseline_model import CLASS_NAMES
class_names_list = [CLASS_NAMES[i] for i in range(8)]

print("\n[*] Generando SHAP Summary Plot — Paleta daltonismo-friendly...")
df_shap = pd.DataFrame(X_shap, columns=feature_names) \
          if feature_names is not None else pd.DataFrame(X_shap)

# SHAP genera el plot con sus colores por defecto
fig, ax = plt.subplots(figsize=(12, 9))
shap.summary_plot(
    shap_values,
    df_shap,
    plot_type    = "bar",
    class_names  = class_names_list,
    max_display  = 20,
    show         = False,
)

# Parchear colores — SHAP usa barh, recoloreamos cada grupo de barras
ax = plt.gca()
bars = [b for b in ax.patches if isinstance(b, plt.Rectangle)]

# SHAP dibuja n_classes barras por feature, en orden de clase
n_classes = 8
for i, bar in enumerate(bars):
    class_idx = i % n_classes
    bar.set_facecolor(WONG_PALETTE[class_idx])
    bar.set_edgecolor('white')
    bar.set_linewidth(0.4)

# Reemplazar leyenda con la nuestra explícita
legend_patches = [
    mpatches.Patch(facecolor=WONG_PALETTE[i],
                   edgecolor='grey', linewidth=0.5,
                   label=f"{i} — {class_names_list[i]}")
    for i in range(8)
]
ax.legend(
    handles      = legend_patches,
    title        = "Clase",
    title_fontsize = 10,
    fontsize     = 9,
    loc          = 'lower right',
    framealpha   = 0.9,
)

plt.title("SHAP Feature Importance — Impacto medio por clase (Top 20)",
          pad=20, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lgbm_shap_summary_bar_accessible.png",
            dpi=300, bbox_inches='tight')
plt.show()
print("[✔] Plot accesible guardado.")

## **6. Guardado del Modelo Final**
Guardamos el objeto del modelo entrenado. Este archivo .pkl será el objetivo a batir por la Tabular ResNet.

In [ ]:
model_path = OUTPUT_DIR / "lgbm_baseline.pkl"
model.save(str(model_path))

print("="*60)
print("🎯 ENTRENAMIENTO LIGHTGBM BASELINE — COMPLETADO")
print("="*60)
print(f"  Test F1-macro : {result_test.f1_macro:.4f}  (producción 95/5)")
print(f"  Test AUC-ROC  : {result_test.auc_roc:.4f}")
print(f"  Modelo guardado en : {model_path}")
print("="*60)